# Pillar 2: Distributional Effect

Model:
income_it = beta1*mgnrega_it + beta2*(mgnrega_it*scst_share_it) + beta3*(mgnrega_it*women_share_it) + rainfall + FE + epsilon

Primary metrics: beta2 and beta3

In [1]:
from pathlib import Path
import pandas as pd
from analysis_utils import load_and_preprocess, fit_fe_clustered, minmax_normalize, compute_vif, summarize_model

DATA_PATH = Path('Panel_Data 2014-24.csv')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

df = load_and_preprocess(str(DATA_PATH))
df['mgnrega_x_scst'] = df['mgnrega'] * df['scst_share']
df['mgnrega_x_women'] = df['mgnrega'] * df['women_share']

In [2]:
formula = 'income ~ mgnrega + mgnrega:scst_share + mgnrega:women_share + rainfall + C(region_id) + C(year)'
results, used_df = fit_fe_clustered(formula, df)
print(results.summary())

coef_table = summarize_model(results, ['mgnrega', 'mgnrega:scst_share', 'mgnrega:women_share', 'rainfall'])
coef_table.to_csv(OUT_DIR / 'pillar2_coefficients.csv', index=False)
print(coef_table)

vif = compute_vif(used_df, ['mgnrega', 'scst_share', 'women_share', 'mgnrega_x_scst', 'mgnrega_x_women', 'rainfall'])
vif.to_csv(OUT_DIR / 'pillar2_vif.csv', index=False)
print(vif)

b2 = results.params.get('mgnrega:scst_share', 0.0)
b3 = results.params.get('mgnrega:women_share', 0.0)
region_stats = used_df.groupby('region_id')[['scst_share', 'women_share', 'mgnrega']].mean()
region_raw = region_stats['mgnrega'] * ((b2 * region_stats['scst_share'] + b3 * region_stats['women_share']) / 2.0)
pillar2_score = minmax_normalize(region_raw).rename('pillar2_distribution_score').reset_index()
pillar2_score.to_csv(OUT_DIR / 'pillar2_distribution_score.csv', index=False)
pillar2_score.head()

                            OLS Regression Results                            
Dep. Variable:                 income   R-squared:                       0.685
Model:                            OLS   Adj. R-squared:                  0.649
Method:                 Least Squares   F-statistic:                     76.19
Date:                Mon, 13 Apr 2026   Prob (F-statistic):           1.53e-82
Time:                        22:34:06   Log-Likelihood:                -25042.
No. Observations:                2560   AIC:                         5.062e+04
Df Residuals:                    2291   BIC:                         5.219e+04
Df Model:                         268                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

C:\Users\Tanishq op\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 268, but rank is 14
  warnings.warn('covariance of constraints does not have full '


,region_id,pillar2_distribution_score
0,ADILABAD_TELANGANA,0.516435
1,AGRA_UTTAR PRADESH,0.307523
2,AJMER_RAJASTHAN,0.854351
3,ALAPPUZHA_KERALA,0.732177
4,ALIGARH_UTTAR PRADESH,0.249480
